# Integrate topic discovery with the LangGraph agent

Worktree-local review notebook. It imports `src/playbook` and walks:

```text
weekly batch
→ DS topic/intent discovery
→ DS subflow discovery
→ DS action-path discovery
→ normalized agent state
→ current playbook/RAG lookup
→ agent decision / HITL
```

Ownership is unchanged:

* **Agent** owns orchestration, `MetaAgentState`, KB/playbook loading, RAG, tools, HITL, persistence.
* **DS** owns `discover_intent_topics`, `discover_subflow_topics`, `discover_action_paths`.
* `playbook.adapters` is the only boundary. BERTopic objects do not enter graph state.

This notebook does **not** hide mismatches.

In [1]:
from __future__ import annotations

import json

from IPython.display import Markdown, display

from playbook import (
    build_graph,
    configure_runtime,
    invoke_week,
    load_playbook,
    render_mermaid,
    retrieve_guidance,
)
from playbook import runtime as rt
from playbook.actions import discover_action_paths
from playbook.adapters import RUNTIME_TOPIC_CONFIG, tasks_to_conversations
from playbook.data import load_week
from playbook.kb import DEFAULT_DATA_DIR
from playbook.schemas import DiscoveredTopic
from playbook.topics import BertopicConfig, discover_intent_topics

playbook, store = configure_runtime(
    data_dir=DEFAULT_DATA_DIR,
    store_path=DEFAULT_DATA_DIR / "integrate_run_store.sqlite",
)


def show(obj, title: str = "") -> None:
    if title:
        display(Markdown(f"**{title}**"))
    display(Markdown(f"```json\n{json.dumps(obj, indent=2, default=str)}\n```"))

## Mismatches left visible

| Area | DS side | Agent side | Adapter choice |
|------|---------|------------|----------------|
| Weekly batch | `load_week("week_1")` from `data/demo/weeks.parquet` | `configure_runtime` + `scratch_data/incoming_conversations.json` | Demo HITL still uses the agent cohort. DS week is shown separately. |
| Conversation | `ConversationRecord` (no labels) | `TaskStore` rows still have `hidden_flow` / `hidden_subflow` | `tasks_to_conversations` drops those fields. |
| Topic output | `TopicInfo` + `TopicDiscoveryResult` (memberships, `cohesive_enough`, `BertopicConfig`) | Plan-frozen `DiscoveredTopic` | Graph state stores `DiscoveredTopic.model_dump()` only. |
| Cluster size | Defaults `min_to_cluster=5`, `min_topic_n=5` | Demo leftover sets are often 3–4 conversations | Runtime uses `RUNTIME_TOPIC_CONFIG` (min 2). |
| Retrieval | `config.KB_JSON` → `data/raw/kb.json` (path constant only) | `load_playbook` / `retrieve_guidance` over `scratch_data/seed_*.json` | Agent KB is the only runtime retriever. |
| Action paths | Counts every conversation | Pathway node evaluates successful traces, then reads DS paths | Intentional: DS function is called; success filter stays on the agent. |
| Labels | Never produced by BERTopic | `propose_*_label` + `simulate_hitl` still name/accept proposals | Discovery ≠ naming. |

In [2]:
ds_defaults = BertopicConfig()
show(
    {
        "ds_default_min_to_cluster": ds_defaults.min_to_cluster,
        "ds_default_min_topic_n": ds_defaults.min_topic_n,
        "runtime_min_to_cluster": RUNTIME_TOPIC_CONFIG.min_to_cluster,
        "runtime_min_topic_n": RUNTIME_TOPIC_CONFIG.min_topic_n,
        "agent_kb_dir": str(DEFAULT_DATA_DIR),
        "agent_kb_files": ["seed_ontology.json", "seed_kb.json", "seed_guidelines.json"],
        "ds_raw_kb_constant": "playbook.config.KB_JSON (not used at runtime)",
    },
    "integration choices",
)

**integration choices**

```json
{
  "ds_default_min_to_cluster": 5,
  "ds_default_min_topic_n": 5,
  "runtime_min_to_cluster": 2,
  "runtime_min_topic_n": 2,
  "agent_kb_dir": "C:\\Users\\doste\\.cursor\\worktrees\\integrate-topic-agent-7d2e9c4a\\scratch_data",
  "agent_kb_files": [
    "seed_ontology.json",
    "seed_kb.json",
    "seed_guidelines.json"
  ],
  "ds_raw_kb_constant": "playbook.config.KB_JSON (not used at runtime)"
}
```

## 1. Weekly batch

Two batch sources exist after the merge. The agent path uses the scratch cohort
so the existing HITL demo still runs. The DS parquet week is unlabeled.

In [3]:
ds_week = load_week("week_1")
agent_tasks = store.fetchall("SELECT task_id, hidden_flow, hidden_subflow, success FROM tasks")
agent_batch = tasks_to_conversations(store.get_tasks([row["task_id"] for row in agent_tasks]))

show(
    {
        "ds_week_id": ds_week.week_id,
        "ds_conversation_ids": ds_week.conversation_ids,
        "ds_record_fields": sorted(ds_week.conversations[0].model_dump()),
        "agent_task_ids": [row["task_id"] for row in agent_tasks],
        "agent_store_still_has_hidden_labels": bool(agent_tasks[0].get("hidden_flow")),
        "adapted_record_fields": sorted(agent_batch[0].model_dump()),
    },
    "batch",
)
assert set(agent_batch[0].model_dump()) == {"conversation_id", "turns", "actions"}

**batch**

```json
{
  "ds_week_id": "week_1",
  "ds_conversation_ids": [
    "1226",
    "169",
    "1821",
    "1877",
    "194",
    "1944",
    "264",
    "450",
    "481",
    "542",
    "587",
    "653",
    "695",
    "717",
    "729",
    "830",
    "888",
    "936"
  ],
  "ds_record_fields": [
    "actions",
    "conversation_id",
    "turns"
  ],
  "agent_task_ids": [
    "u1",
    "p1",
    "t1",
    "t2",
    "t3",
    "l1",
    "l2",
    "l3",
    "s1",
    "s2",
    "s3",
    "x1"
  ],
  "agent_store_still_has_hidden_labels": true,
  "adapted_record_fields": [
    "actions",
    "conversation_id",
    "turns"
  ]
}
```

## 2–4. DS discovery functions (callable, not a new architecture)

`fit_topic_model` is stubbed here so the notebook stays fast and deterministic.
The real `discover_*` wrappers still run. Production nodes call the same wrappers.

In [4]:
from playbook import topics as topic_mod

original_fit = topic_mod.fit_topic_model


def notebook_fit(documents, config, **kwargs):
    topic_ids = []
    descriptors = {}
    for document in documents:
        text = document.lower()
        if any(token in text for token in ("package", "shipment", "delivered", "porch", "carrier")):
            topic_ids.append(0)
            descriptors[0] = "package, missing, delivered"
        elif any(token in text for token in ("two-factor", "authenticator")):
            topic_ids.append(0)
            descriptors[0] = "two-factor, authenticator, reset"
        elif any(token in text for token in ("locked", "lockout")):
            topic_ids.append(1)
            descriptors[1] = "account, locked, lockout"
        else:
            topic_ids.append(-1)
            descriptors.setdefault(-1, "")
    from playbook.topics import FittedTopics

    return FittedTopics(topic_ids=topic_ids, descriptors=descriptors)


topic_mod.fit_topic_model = notebook_fit
print("fit_topic_model stubbed:", topic_mod.fit_topic_model is not original_fit)
print("discover_intent_topics still the package function:", discover_intent_topics)

direct_intents = discover_intent_topics(agent_batch, config=RUNTIME_TOPIC_CONFIG)
direct_paths = discover_action_paths(agent_batch)
show([topic.model_dump() for topic in direct_intents.topics], "direct DS intent topics (includes noise topic -1)")
show(direct_paths.model_dump(), "direct DS action paths")

fit_topic_model stubbed: True
discover_intent_topics still the package function: <function discover_intent_topics at 0x000002D6E7A271A0>


**direct DS intent topics (includes noise topic -1)**

```json
[
  {
    "topic_id": -1,
    "size": 3,
    "descriptor": "",
    "member_ids": [
      "u1",
      "p1",
      "x1"
    ],
    "representative_ids": [
      "u1",
      "p1",
      "x1"
    ],
    "cohesive_enough": false,
    "parent_intent": null
  },
  {
    "topic_id": 0,
    "size": 6,
    "descriptor": "package, missing, delivered",
    "member_ids": [
      "t1",
      "t2",
      "t3",
      "s1",
      "s2",
      "s3"
    ],
    "representative_ids": [
      "t1",
      "t2",
      "t3"
    ],
    "cohesive_enough": true,
    "parent_intent": null
  },
  {
    "topic_id": 1,
    "size": 3,
    "descriptor": "account, locked, lockout",
    "member_ids": [
      "l1",
      "l2",
      "l3"
    ],
    "representative_ids": [
      "l1",
      "l2",
      "l3"
    ],
    "cohesive_enough": true,
    "parent_intent": null
  }
]
```

**direct DS action paths**

```json
{
  "n_conversations": 12,
  "n_unique_paths": 7,
  "paths": [
    {
      "actions": [
        "pull-up-account",
        "enter-details",
        "send-link"
      ],
      "count": 3,
      "conversation_ids": [
        "t1",
        "t2",
        "t3"
      ]
    },
    {
      "actions": [
        "pull-up-account",
        "validate-purchase",
        "record-reason",
        "update-order",
        "make-purchase"
      ],
      "count": 3,
      "conversation_ids": [
        "s1",
        "s2",
        "s3"
      ]
    },
    {
      "actions": [],
      "count": 2,
      "conversation_ids": [
        "l1",
        "x1"
      ]
    },
    {
      "actions": [
        "notify-team"
      ],
      "count": 1,
      "conversation_ids": [
        "l2"
      ]
    },
    {
      "actions": [
        "pull-up-account",
        "enter-details",
        "make-password"
      ],
      "count": 1,
      "conversation_ids": [
        "p1"
      ]
    },
    {
      "actions": [
        "pull-up-account",
        "verify-identity"
      ],
      "count": 1,
      "conversation_ids": [
        "u1"
      ]
    },
    {
      "actions": [
        "try-again"
      ],
      "count": 1,
      "conversation_ids": [
        "l3"
      ]
    }
  ],
  "button_counts": {
    "pull-up-account": 8,
    "enter-details": 4,
    "send-link": 3,
    "validate-purchase": 3,
    "make-purchase": 3,
    "update-order": 3,
    "record-reason": 3,
    "verify-identity": 1,
    "make-password": 1,
    "notify-team": 1,
    "try-again": 1
  }
}
```

## 5–7. Real agent path: graph state, RAG, decision

`invoke_week` still uses the existing meta-agent:
`establish_cohort → classify → discover → recommend → summarize`.
Discover nodes now call the DS functions and `retrieve_guidance`.

In [5]:
result = invoke_week(
    "week-2026-09-01",
    {
        "source": "scratch_data/incoming_conversations.json",
        "window": "demo-week",
        "as_of": "2026-09-01",
    },
)

show(
    {
        "run_id": result["run_id"],
        "current_stage": result["current_stage"],
        "kb_version": result["kb_version"],
        "discovered_topics": result["discovered_topics"],
        "discovered_subflows": result["discovered_subflows"],
        "action_paths": result["action_paths"],
        "retrieved_guidance": result["retrieved_guidance"],
        "discovery_summary": result["discovery_summary"],
        "recommendation_summary": result["recommendation_summary"],
    },
    "normalized graph state",
)

for row in result["discovered_topics"]:
    DiscoveredTopic.model_validate(row)
for row in result["discovered_subflows"]:
    DiscoveredTopic.model_validate(row)

proposals = [
    {
        "proposal_type": row["proposal_type"],
        "candidate": row["candidate"],
        "review_decision": row["review_decision"],
        "parent_intent": row["parent_intent"],
    }
    for row in rt.store.list_proposals("week-2026-09-01")
]
show(proposals, "HITL decisions")
show(rt.store.list_recommendations("week-2026-09-01"), "pathway recommendations")

**normalized graph state**

```json
{
  "run_id": "week-2026-09-01",
  "current_stage": "summarize",
  "kb_version": 6,
  "discovered_topics": [
    {
      "topic_id": 0,
      "size": 3,
      "descriptor": "package, missing, delivered",
      "representative_conversation_ids": [
        "s1",
        "s2",
        "s3"
      ]
    }
  ],
  "discovered_subflows": [
    {
      "topic_id": 0,
      "size": 3,
      "descriptor": "two-factor, authenticator, reset",
      "representative_conversation_ids": [
        "t1",
        "t2",
        "t3"
      ]
    },
    {
      "topic_id": 1,
      "size": 3,
      "descriptor": "account, locked, lockout",
      "representative_conversation_ids": [
        "l1",
        "l2",
        "l3"
      ]
    },
    {
      "topic_id": 0,
      "size": 3,
      "descriptor": "package, missing, delivered",
      "representative_conversation_ids": [
        "s1",
        "s2",
        "s3"
      ]
    }
  ],
  "action_paths": [
    {
      "actions": [
        "pull-up-account",
        "enter-details",
        "send-link"
      ],
      "count": 3,
      "conversation_ids": [
        "t1",
        "t2",
        "t3"
      ]
    },
    {
      "actions": [
        "pull-up-account",
        "validate-purchase",
        "record-reason",
        "update-order",
        "make-purchase"
      ],
      "count": 3,
      "conversation_ids": [
        "s1",
        "s2",
        "s3"
      ]
    }
  ],
  "retrieved_guidance": [
    {
      "topic_id": 0,
      "query": "package, missing, delivered",
      "hits": [
        {
          "intent_id": "account_access",
          "score": 0.0,
          "doc": "Account Access account_access username, password, two-factor authentication, and other login / account-access problems including lockouts Recover Username Recover Password recover_username recover_password",
          "subflows": [
            "recover_username",
            "recover_password"
          ]
        }
      ]
    },
    {
      "topic_id": 0,
      "query": "two-factor, authenticator, reset",
      "hits": [
        {
          "intent_id": "account_access",
          "score": 0.125,
          "doc": "Account Access account_access username, password, two-factor authentication, and other login / account-access problems including lockouts Recover Username Recover Password recover_username recover_password",
          "subflows": [
            "recover_username",
            "recover_password"
          ]
        },
        {
          "intent_id": "shipping_issue",
          "score": 0.0,
          "doc": "Shipping Issue shipping_issue package shipment delivered missing items carrier porch tracking order purchase replacement",
          "subflows": []
        }
      ]
    },
    {
      "topic_id": 1,
      "query": "account, locked, lockout",
      "hits": [
        {
          "intent_id": "account_access",
          "score": 0.062,
          "doc": "Account Access account_access username, password, two-factor authentication, and other login / account-access problems including lockouts Recover Username Recover Password recover_username recover_password",
          "subflows": [
            "recover_username",
            "recover_password"
          ]
        },
        {
          "intent_id": "shipping_issue",
          "score": 0.0,
          "doc": "Shipping Issue shipping_issue package shipment delivered missing items carrier porch tracking order purchase replacement",
          "subflows": []
        }
      ]
    },
    {
      "topic_id": 0,
      "query": "package, missing, delivered",
      "hits": [
        {
          "intent_id": "shipping_issue",
          "score": 0.231,
          "doc": "Shipping Issue shipping_issue package shipment delivered missing items carrier porch tracking order purchase replacement",
          "subflows": []
        },
        {
          "intent_id": "account_access",
          "score": 0.0,
          "doc": "Account Access account_access username, password, two-factor authentication, and other login / account-access problems including lockouts Recover Username Recover Password recover_username recover_password",
          "subflows": [
            "recover_username",
            "recover_password"
          ]
        }
      ]
    }
  ],
  "discovery_summary": {
    "intents": {
      "unresolved_tasks": 4,
      "candidate_count": 1,
      "outlier_count": 1,
      "approved_count": 1,
      "rejected_count": 1
    },
    "subflows": {
      "unresolved_tasks": 9,
      "candidate_count": 2,
      "emerging_count": 1,
      "approved_count": 2,
      "rejected_count": 1
    }
  },
  "recommendation_summary": {
    "items": [
      {
        "subflow_id": "reset_2fa",
        "intent_id": "account_access",
        "supported": true,
        "n_tasks": 3,
        "support": 1.0
      },
      {
        "subflow_id": "missing",
        "intent_id": "shipping_issue",
        "supported": true,
        "n_tasks": 3,
        "support": 1.0
      }
    ],
    "recommended": 2
  }
}
```

**HITL decisions**

```json
[
  {
    "proposal_type": "outlier",
    "candidate": null,
    "review_decision": "decline",
    "parent_intent": null
  },
  {
    "proposal_type": "new_intent",
    "candidate": "shipping_issue",
    "review_decision": "accept",
    "parent_intent": null
  },
  {
    "proposal_type": "new_subflow",
    "candidate": "reset_2fa",
    "review_decision": "accept",
    "parent_intent": "account_access"
  },
  {
    "proposal_type": "emerging",
    "candidate": null,
    "review_decision": "decline",
    "parent_intent": "account_access"
  },
  {
    "proposal_type": "new_subflow",
    "candidate": "missing",
    "review_decision": "accept",
    "parent_intent": "shipping_issue"
  }
]
```

**pathway recommendations**

```json
[
  {
    "rec_id": "rec_a66547d2",
    "run_id": "week-2026-09-01",
    "intent_id": "account_access",
    "subflow_id": "reset_2fa",
    "supporting_task_ids": "[\"t1\", \"t2\", \"t3\"]",
    "kb_draft": "{\"reset_2fa\": [\"pull-up-account\", \"enter-details\", \"send-link\"]}",
    "guideline_draft": "{\"Account Access\": {\"subflows\": {\"Reset Two-Factor Auth\": {\"actions\": [{\"type\": \"interaction\", \"button\": \"Pull Up Account\", \"text\": \"Observed repeated action [pull-up-account]\", \"subtext\": []}, {\"type\": \"interaction\", \"button\": \"Enter Details\", \"text\": \"Observed repeated action [enter-details]\", \"subtext\": []}, {\"type\": \"interaction\", \"button\": \"Send Link\", \"text\": \"Observed repeated action [send-link]\", \"subtext\": []}], \"instructions\": [\"Inferred from repeated successful traces in this batch.\", \"Needs human review before it becomes live playbook text.\"]}}}}",
    "evaluation": "{\"supported\": true, \"successful\": 3, \"support\": 1.0}",
    "created_at": "2026-09-04T00:39:31.715331+00:00"
  },
  {
    "rec_id": "rec_33fd9354",
    "run_id": "week-2026-09-01",
    "intent_id": "shipping_issue",
    "subflow_id": "missing",
    "supporting_task_ids": "[\"s1\", \"s2\", \"s3\"]",
    "kb_draft": "{\"missing\": [\"pull-up-account\", \"validate-purchase\", \"record-reason\", \"update-order\", \"make-purchase\"]}",
    "guideline_draft": "{\"Shipping Issue\": {\"subflows\": {\"Missing Item\": {\"actions\": [{\"type\": \"interaction\", \"button\": \"Pull Up Account\", \"text\": \"Observed repeated action [pull-up-account]\", \"subtext\": []}, {\"type\": \"interaction\", \"button\": \"Validate Purchase\", \"text\": \"Observed repeated action [validate-purchase]\", \"subtext\": []}, {\"type\": \"interaction\", \"button\": \"Record Reason\", \"text\": \"Observed repeated action [record-reason]\", \"subtext\": []}, {\"type\": \"interaction\", \"button\": \"Update Order\", \"text\": \"Observed repeated action [update-order]\", \"subtext\": []}, {\"type\": \"interaction\", \"button\": \"Make Purchase\", \"text\": \"Observed repeated action [make-purchase]\", \"subtext\": []}], \"instructions\": [\"Inferred from repeated successful traces in this batch.\", \"Needs human review before it becomes live playbook text.\"]}}}}",
    "evaluation": "{\"supported\": true, \"successful\": 3, \"support\": 1.0}",
    "created_at": "2026-09-04T00:39:31.908039+00:00"
  }
]
```

## Canonical KB / playbook path

One retriever: `playbook.kb.retrieve_guidance` over `scratch_data` seed files.

In [6]:
seed = load_playbook(DEFAULT_DATA_DIR)
manual_hits = retrieve_guidance("package missing delivered", seed)
show(
    {
        "retriever": "playbook.kb.retrieve_guidance",
        "data_dir": str(DEFAULT_DATA_DIR),
        "seed_intents": seed.intent_ids(),
        "seed_subflows": seed.ontology["intents"]["subflows"],
        "manual_hits": manual_hits,
        "graph_used_same_retriever": bool(result["retrieved_guidance"]),
        "live_intents_after_hitl": rt.playbook.intent_ids(),
        "live_subflows_after_hitl": rt.playbook.ontology["intents"]["subflows"],
    },
    "canonical KB",
)

**canonical KB**

```json
{
  "retriever": "playbook.kb.retrieve_guidance",
  "data_dir": "C:\\Users\\doste\\.cursor\\worktrees\\integrate-topic-agent-7d2e9c4a\\scratch_data",
  "seed_intents": [
    "account_access"
  ],
  "seed_subflows": {
    "account_access": [
      "recover_username",
      "recover_password"
    ]
  },
  "manual_hits": [
    {
      "intent_id": "account_access",
      "score": 0.0,
      "doc": "Account Access account_access username, password, two-factor authentication, and other login / account-access problems including lockouts Recover Username Recover Password recover_username recover_password",
      "subflows": [
        "recover_username",
        "recover_password"
      ]
    }
  ],
  "graph_used_same_retriever": true,
  "live_intents_after_hitl": [
    "account_access",
    "shipping_issue"
  ],
  "live_subflows_after_hitl": {
    "account_access": [
      "recover_username",
      "recover_password",
      "reset_2fa"
    ],
    "shipping_issue": [
      "missing"
    ]
  }
}
```

## Mermaid (compiled subgraphs, x-ray)

In [7]:
source = render_mermaid(build_graph(True), xray=1)
display(Markdown(f"```mermaid\n{source}\n```"))
print("graph name:", build_graph(False).name)
print("mermaid has discover:", "discover" in source)
print("mermaid has recommend:", "recommend" in source)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	establish_cohort(establish_cohort<hr/><small><em>agent = meta
phase = query</em></small>)
	recommend(recommend<hr/><small><em>agent = pathway
phase = process</em></small>)
	summarize(summarize<hr/><small><em>agent = meta
phase = summarize</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> establish_cohort;
	classify\3aclassify_subflows --> discover\3adiscover_intents;
	discover\3areclassify_subflows --> recommend;
	establish_cohort --> classify\3aclassify_intents;
	recommend --> summarize;
	summarize --> __end__;
	subgraph classify
	classify\3aclassify_intents(classify_intents<hr/><small><em>agent = intent
phase = process</em></small>)
	classify\3aclassify_subflows(classify_subflows<hr/><small><em>agent = subflow
phase = process</em></small>)
	classify\3aclassify_intents --> classify\3aclassify_subflows;
	end
	subgraph discover
	discover\3adiscover_intents(discover_intents<hr/><small><em>agent = intent_discovery
phase = process</em></small>)
	discover\3areclassify_intents(reclassify_intents<hr/><small><em>agent = intent
phase = process</em></small>)
	discover\3adiscover_subflows(discover_subflows<hr/><small><em>agent = subflow_discovery
phase = process</em></small>)
	discover\3areclassify_subflows(reclassify_subflows<hr/><small><em>agent = subflow
phase = process</em></small>)
	discover\3adiscover_intents --> discover\3areclassify_intents;
	discover\3adiscover_subflows --> discover\3areclassify_subflows;
	discover\3areclassify_intents --> discover\3adiscover_subflows;
	end
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

graph name: meta_agent
mermaid has discover: True
mermaid has recommend: True


In [8]:
topic_mod.fit_topic_model = original_fit
print("restored fit_topic_model")

restored fit_topic_model
